# S03 · Grouping and joining tables

Two everyday moves on tables. First, group the rows to get totals for each customer
(this is how the shop owner sees her big spenders). Second, join a second sheet onto
the first when the information you need is spread across two tables.

**New here? Read this once.**

- New to Python? Run each cell top to bottom with the play button and read the note
  above it. Nothing here is mathematically hard; it is a small vocabulary of table
  moves.
- Used a pivot table in Excel? `groupby` is that idea. Used VLOOKUP? `merge` is
  that idea. You already know the moves; this is the code for them.
- Already comfortable? Look for the cell marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [1]:
# This notebook only uses pandas, which Google Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

Setup complete - nothing to install.


In [2]:
import pandas as pd   # pandas is the library for working with tables

## Step 1 — a small orders table

Each row is one order: which customer placed it, in which city, and the amount they
spent in rupees. We build it in code so we know exactly what is in it. Notice some
customers appear more than once, because they ordered more than once.

In [3]:
# Each key is a column; each list goes down that column.
orders_data = {
    "customer": ["Asha", "Ravi", "Asha", "Meera", "Ravi", "Asha"],
    "city":     ["Pune", "Pune", "Pune", "Mumbai", "Pune", "Pune"],
    "amount":   [250, 180, 300, 120, 200, 150],   # rupees
}

orders = pd.DataFrame(orders_data)
print(orders)

  customer    city  amount
0     Asha    Pune     250
1     Ravi    Pune     180
2     Asha    Pune     300
3    Meera  Mumbai     120
4     Ravi    Pune     200
5     Asha    Pune     150


## Step 2 — total the spend per customer (split-apply-combine)

**groupby** is the idea of *split-apply-combine*, and it is exactly what a pivot
table does:

1. **split** the rows into groups (here, one group per customer),
2. **apply** a calculation to each group (here, add up the amounts),
3. **combine** the answers into a small result table.

Let's find the total amount each customer spent.

In [4]:
# Group the rows by customer, then sum the "amount" column inside each group.
total_per_customer = orders.groupby("customer")["amount"].sum()

print("total amount spent by each customer (rupees):")
print(total_per_customer)

total amount spent by each customer (rupees):
customer
Asha     700
Meera    120
Ravi     380
Name: amount, dtype: int64


## Step 3 — other summaries on the same groups

Once the rows are grouped you can ask for different calculations. Here we count how
many orders each customer made, and find their average order size. Same split, a
different calculation applied to each group.

In [5]:
# How many orders did each customer make? (count the rows in each group)
orders_per_customer = orders.groupby("customer")["amount"].count()
print("number of orders per customer:")
print(orders_per_customer)

# What was each customer's average order amount?
print()
average_per_customer = orders.groupby("customer")["amount"].mean()
print("average order amount per customer (rupees):")
print(average_per_customer)

number of orders per customer:
customer
Asha     3
Meera    1
Ravi     2
Name: amount, dtype: int64

average order amount per customer (rupees):
customer
Asha     233.333333
Meera    120.000000
Ravi     190.000000
Name: amount, dtype: float64


## Step 4 — a second sheet to join in

Often the fact you need lives in a different sheet. Here is a second small table
giving each customer's loyalty tier. The shared column `customer` is the **key**
that links the two tables, the way a common ID links two sheets in a workbook.

In [6]:
# A separate table: one row per customer, with their loyalty tier.
tiers_data = {
    "customer": ["Asha", "Ravi", "Meera"],
    "tier":     ["gold", "silver", "bronze"],
}

tiers = pd.DataFrame(tiers_data)
print(tiers)

  customer    tier
0     Asha    gold
1     Ravi  silver
2    Meera  bronze


## Step 5 — join the two tables on the key (`merge`)

**merge** glues the two tables together by matching rows that share the same
`customer`. Every order row gets its customer's tier added on. If you have used
VLOOKUP in Excel to pull a value from another sheet, this is the same idea, done
for the whole table at once.

In [7]:
# Join orders and tiers by matching the "customer" column.
orders_with_tier = pd.merge(orders, tiers, on="customer")

print("BEFORE - orders had no tier column:")
print(orders)
print()
print("AFTER - each order now also shows the customer's tier:")
print(orders_with_tier)

BEFORE - orders had no tier column:
  customer    city  amount
0     Asha    Pune     250
1     Ravi    Pune     180
2     Asha    Pune     300
3    Meera  Mumbai     120
4     Ravi    Pune     200
5     Asha    Pune     150

AFTER - each order now also shows the customer's tier:
  customer    city  amount    tier
0     Asha    Pune     250    gold
1     Ravi    Pune     180  silver
2     Asha    Pune     300    gold
3    Meera  Mumbai     120  bronze
4     Ravi    Pune     200  silver
5     Asha    Pune     150    gold


## Step 6 — group the joined table

Now that a tier sits on every row, we can group by tier instead of by customer. The
two moves combine naturally: join first, then summarise. This answers a manager's
question directly: how much revenue comes from each loyalty tier?

In [8]:
# Total amount spent by customers in each loyalty tier.
total_per_tier = orders_with_tier.groupby("tier")["amount"].sum()

print("total amount spent per loyalty tier (rupees):")
print(total_per_tier)

total amount spent per loyalty tier (rupees):
tier
bronze    120
gold      700
silver    380
Name: amount, dtype: int64


### Stretch (optional) — several summaries in one table

Skip this if the basics are still settling. If you want the tidy professional
version, `agg` lets you ask for several summaries at once and lays them out in one
small table. Here we get each customer's total, their average, and their order
count together, instead of computing them one at a time as we did above.

In [9]:
# Ask for three summaries of the "amount" column, per customer, in one go.
summary_per_customer = orders.groupby("customer")["amount"].agg(["sum", "mean", "count"])

print("total, average and order count per customer, in one table:")
print(summary_per_customer)

total, average and order count per customer, in one table:
          sum        mean  count
customer                        
Asha      700  233.333333      3
Meera     120  120.000000      1
Ravi      380  190.000000      2


## What you just did

You used **groupby** to split rows into groups and summarise each group
(split-apply-combine, the pivot-table idea), and **merge** to join two tables on a
shared key (the VLOOKUP idea). Almost every business report is built from these two
moves. The shop owner's "big spenders" and "revenue per tier" questions are now one
line each.

Next notebook: `03_clean_a_messy_excel_file.ipynb`, where we face real, messy Excel
data and clean it end to end.